# AASHTO Vessel Allision Risk Determination

$$ AF = (N)(PA)(PG)(PC)(PF) $$

$ N $ = Annual Number of Vessels  
$ PA $ = Probability of Aberrancy  
$ PG $ = Geometric Probability  
$ PC $ = Probability of Collapse  
$ PF $ = Protection Factor  

AASHTO Recommends $ AF < 0.0001 $ for critical bridges and $ AF < 0.001 $ for  
regular bridges.

# $N$ - Number of Vessels

This seems to basically be the only value that ultimately ends up mattering.  

Basically the bridges this is identifying are highly trafficed bridges  
without pier protection.

This script calculates $N$, the annual frequency of vessels classified by  
type, size, and loading condition.

### Applicable Codes:
* **AASHTO LRFD Bridge Design Specifications (10th Ed, 2024):**  
Section 3.14.5.1  
* **AASHTO Guide Specifications (2009):** Section 4.2  

### Methodology:
1.  **Ingest Fleet Data:** Load raw vessel data (e.g., from USACE  
Waterborne Commerce Statistics).
2.  **Classify:** Group vessels into AASHTO standard DWT (Deadweight  
Tonnage) bins.
3.  **Filter:** Remove vessels that cannot physically reach the bridge due  
to draft restrictions.
4.  **Project Growth:** Apply a Growth Factor ($GF$) to estimate traffic at  
the bridge's mid-life.
5.  **Summarize:** Output the final $N$ table for use in the Probability of  
Collapse ($PC$) calculation.

In [2]:
import pandas as pd
import numpy as np

# --- CONFIGURATION PARAMETERS (AASHTO 3.14.5.1) ---

# 1. Bridge & Waterway Constraints
CHANNEL_DEPTH_MLLW = 45.0  # ft (Mean Lower Low Water)
TIDE_ALLOWANCE = 2.0       # ft (Safety buffer or high tide consideration)
MAX_DRAFT_ALLOWED = CHANNEL_DEPTH_MLLW + TIDE_ALLOWANCE

# 2. Growth Factor (GF)
CURRENT_YEAR = 2024
BRIDGE_MID_LIFE_YEAR = 2074 # Typically 50 years from opening
ANNUAL_GROWTH_RATE = 0.015  # 1.5% annual growth

# Calculate Compounded Growth Factor
years_growth = BRIDGE_MID_LIFE_YEAR - CURRENT_YEAR
GF = (1 + ANNUAL_GROWTH_RATE) ** years_growth

print(f"Max Allowable Draft: {MAX_DRAFT_ALLOWED} ft")
print(f"Growth Factor (GF) for {years_growth} years: {GF:.3f}")



Max Allowable Draft: 47.0 ft
Growth Factor (GF) for 50 years: 2.105


#### Step 1: Define the Vessel Fleet
In a real analysis, you would import a CSV file here. For this example, we  
generate a synthetic fleet representing a mix of Tugs, Barges, and  
Panamax/Post-Panamax Ships.

In [3]:
# Create dummy data representing a year of traffic logs
data = {
    'Vessel_Name': [
        'Tug Hercules', 'Barge 101', 'Handymax A', 'Panamax Star', 
        'Post-Panamax Giant', 'Small Fishing', 'Tanker X', 'Barge 102', 
        'Neo-Panamax Z'
    ],
    'Type': [
        'Tug', 'Barge', 'Bulk Carrier', 'Container', 
        'Container', 'Fishing', 'Tanker', 'Barge', 'Container'
    ],
    'DWT_Tons': [
        500, 3000, 45000, 75000, 
        120000, 50, 60000, 3000, 150000
    ],
    'Draft_ft': [
        10, 12, 35, 42, 
        50, 5, 38, 12, 55
    ],
    'Annual_Trips': [
        200, 150, 45, 30, 
        10, 500, 25, 150, 5
    ]
}

df_fleet = pd.DataFrame(data)
print("--- Initial Fleet Data ---")
print(df_fleet)

--- Initial Fleet Data ---
          Vessel_Name          Type  DWT_Tons  Draft_ft  Annual_Trips
0        Tug Hercules           Tug       500        10           200
1           Barge 101         Barge      3000        12           150
2          Handymax A  Bulk Carrier     45000        35            45
3        Panamax Star     Container     75000        42            30
4  Post-Panamax Giant     Container    120000        50            10
5       Small Fishing       Fishing        50         5           500
6            Tanker X        Tanker     60000        38            25
7           Barge 102         Barge      3000        12           150
8       Neo-Panamax Z     Container    150000        55             5


### Step 2: Filter by Accessibility

AASHTO allows you to exclude vessels that cannot physically hit the pier  
because they would run aground first. We filter out any vessel with a draft  
greater than our `MAX_DRAFT_ALLOWED`.

In [6]:
# Create a 'Passable' column
df_fleet['Can_Navigate'] = df_fleet['Draft_ft'] <= MAX_DRAFT_ALLOWED

# Separate excluded vessels for reporting
excluded_vessels = df_fleet[~df_fleet['Can_Navigate']]
valid_fleet = df_fleet[df_fleet['Can_Navigate']].copy()

print(f"\nExcluded {len(excluded_vessels)} vessel classes due to draft restrictions (Will ground before impact).")
print(valid_fleet[['Vessel_Name', 'DWT_Tons', 'Draft_ft', 'Annual_Trips']])


Excluded 2 vessel classes due to draft restrictions (Will ground before impact).
     Vessel_Name  DWT_Tons  Draft_ft  Annual_Trips
0   Tug Hercules       500        10           200
1      Barge 101      3000        12           150
2     Handymax A     45000        35            45
3   Panamax Star     75000        42            30
5  Small Fishing        50         5           500
6       Tanker X     60000        38            25
7      Barge 102      3000        12           150


### Step 3: Binning by DWT (AASHTO Method II Classifications)

Method II requires analyzing vessels in groups. Calculating $AF$ for every  
single unique ship is inefficient. We group them into **DWT Bins**.

In [7]:
# Define DWT Bins (Example structure, customizable)
bins = [0, 1000, 10000, 40000, 80000, 150000, 1000000]
labels = ['<1k (Small)', '1k-10k (Barge/Tug)', '10k-40k (Small Ship)', '40k-80k (Panamax)', '80k-150k (Post-Panamax)', '>150k (Ultra Large)']

valid_fleet['DWT_Class'] = pd.cut(valid_fleet['DWT_Tons'], bins=bins, labels=labels)

# Group by Class to get N_current
n_summary = valid_fleet.groupby('DWT_Class', observed=True)['Annual_Trips'].sum().reset_index()
n_summary.rename(columns={'Annual_Trips': 'N_current'}, inplace=True)

print("\n--- Fleet Binned by DWT Class ---")
print(n_summary)


--- Fleet Binned by DWT Class ---
            DWT_Class  N_current
0         <1k (Small)        700
1  1k-10k (Barge/Tug)        300
2   40k-80k (Panamax)        100


### Step 4: Apply Growth Factor to get Final $N$
We apply the calculated $GF$ to the current traffic to get the Design $N$.

In [9]:
n_summary['Growth_Factor'] = GF
n_summary['N_design'] = n_summary['N_current'] * n_summary['Growth_Factor']

# Round N_design to nearest whole trip
n_summary['N_design'] = n_summary['N_design'].round(0).astype(int)

# Calculate Probabilities (Optional visualization step: What % of fleet is each class?)
total_N = n_summary['N_design'].sum()
n_summary['Percent_of_Fleet'] = (n_summary['N_design'] / total_N) * 100

# Display Final Table
print("\n--- Final N (Annual Number of Vessels) for Risk Analysis ---")
print(n_summary)


--- Final N (Annual Number of Vessels) for Risk Analysis ---
            DWT_Class  N_current  Growth_Factor  N_design  Percent_of_Fleet
0         <1k (Small)        700       2.105242      1474         63.616746
1  1k-10k (Barge/Tug)        300       2.105242       632         27.276651
2   40k-80k (Panamax)        100       2.105242       211          9.106603


# $PA$ - Probability of Aberrancy

# $PG$ - Geometric Probability

# $PC$ - Probability of Collapse

# $PF$ - Protection Factor